### Interface

In [1]:
import torch.nn as nn
import math
import os
import pickle
import numpy as np
import torch
import pandas as pd 

import warnings

warnings.filterwarnings(
    "ignore",
    message=".*pkg_resources.*",
    category=UserWarning
)

import sys
sys.path.insert(0, "NES-Music-Maker-master")

import nesmdb
import soundfile as sf
import random

import torch.nn.functional as F


def sample_from_logits(logits, temperature=0.8, top_k=10):
    # Temperature scaling
    scaled_logits = logits / temperature

    # Top-k filtering
    if top_k is not None:
        top_k = min(top_k, scaled_logits.size(-1))
        values, _ = torch.topk(scaled_logits, top_k)
        min_value = values[:, -1].unsqueeze(-1)
        scaled_logits = torch.where(
            scaled_logits < min_value,
            torch.full_like(scaled_logits, float('-inf')),
            scaled_logits
        )

    probs = F.softmax(scaled_logits, dim=-1)
    next_token = torch.multinomial(probs, num_samples=1)
    return next_token.item()

In [2]:
class MusicLSTM(nn.Module):
    def __init__(self, 
                 input_size, 
                 hidden_size,  
                 dropout_rate=0.2):
        super().__init__()
        
        # Embed notes so they are vector representations instead of integer values 
        
        self.p1_embedding = nn.Embedding(num_embed_p1, embed_dim_p1)
        self.p2_embedding = nn.Embedding(num_embed_p2, embed_dim_p2)
        self.tr_embedding = nn.Embedding(num_embed_tr, embed_dim_tr)
        self.no_embedding = nn.Embedding(num_embed_no, embed_dim_no)

        self.lstm1 = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True)
        self.norm1 = nn.LayerNorm(hidden_size)
        self.dropout1 = nn.Dropout(dropout_rate)

        self.lstm2 = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size, batch_first=True)
        self.norm2 = nn.LayerNorm(hidden_size)
        self.dropout2 = nn.Dropout(dropout_rate)

        self.attention = nn.MultiheadAttention(embed_dim=hidden_size, num_heads=4, batch_first=True)
        self.norm_attn = nn.LayerNorm(hidden_size)
        
        self.out_p1 = nn.Linear(hidden_size, len(voice_data['p1']))
        self.out_p2 = nn.Linear(hidden_size, len(voice_data['p2']))
        self.out_tr = nn.Linear(hidden_size, len(voice_data['tr']))
        self.out_no = nn.Linear(hidden_size, len(voice_data['no']))

    def forward(self, x):
        
        p1_vectors = self.p1_embedding((x[:, :, 0]))
        p2_vectors = self.p2_embedding((x[:, :, 1]))
        tr_vectors = self.tr_embedding((x[:, :, 2]))
        no_vectors = self.no_embedding((x[:, :, 3]))

        model_input = torch.cat([p1_vectors, p2_vectors, tr_vectors, no_vectors], axis = 2) 
        
        x, _ = self.lstm1(model_input)
        x = self.norm1(x)
        x = self.dropout1(x)

        x, _ = self.lstm2(x)
        x = self.norm2(x)
        x = self.dropout2(x)

        query = x[:, -1:, :]  
        attn_out, attn_weights = self.attention(query, x, x)  
        x = self.norm_attn(query.squeeze(1) + attn_out.squeeze(1))

        pred_p1 = self.out_p1(x)
        pred_p2 = self.out_p2(x)
        pred_tr = self.out_tr(x)
        pred_no = self.out_no(x)

        return pred_p1, pred_p2, pred_tr, pred_no

In [3]:
# Run code from other file to recreate the vector map 

# Train
train_dir = 'nesmdb24_seprsco/train'
train_files = os.listdir(train_dir)

train_data = [] 

for filename in train_files:
    filepath = os.path.join(train_dir, filename)
    with open(filepath, 'rb') as f:
        rate, nsamps, seprsco = pickle.load(f)
    train_data.append(seprsco)

list_lengthhs = []              # Drops songs that are too short or to long to prevent noise and overfitting 
for song in train_data:
    list_lengthhs.append(len(song))
    
arr = np.array(list_lengthhs)
p5 = np.percentile(arr, 5)
p95 = np.percentile(arr, 95)    

train_data = [song for song in train_data if len(song) > p5 and len(song) < p95]

# Get values of the music voices, 8-bit music is split into four parts/voices

voice_names = ['p1', 'p2', 'tr', 'no']
voice_data = {}

for column, name in enumerate(voice_names):
    all_values = np.concatenate([song[:, column] for song in train_data])
    value_index = {val: i for i, val in enumerate(np.unique(all_values))}
    
    voice_data[name] = value_index


num_embed_p1 = len(voice_data['p1'])
num_embed_p2 = len(voice_data['p2'])
num_embed_tr = len(voice_data['tr'])
num_embed_no = len(voice_data['no'])


embed_dim_p1 = math.ceil(len(voice_data['p1']) ** 0.25) 
embed_dim_p2 = math.ceil(len(voice_data['p2']) ** 0.25) 
embed_dim_tr = math.ceil(len(voice_data['tr']) ** 0.25) 
embed_dim_no = math.ceil(len(voice_data['no']) ** 0.25) 

# Map data to a consistant value space 
import pandas as pd

# Train
train_data_mapped = []

for song in train_data:
    
    remapped_song = np.column_stack([pd.Series(song[:,0]).map(voice_data['p1']),
                                     pd.Series(song[:,1]).map(voice_data['p2']), 
                                     pd.Series(song[:,2]).map(voice_data['tr']), 
                                     pd.Series(song[:,3]).map(voice_data['no'])])
    train_data_mapped.append(remapped_song)


In [4]:
# Valid 
valid_dir = 'nesmdb24_seprsco/valid'
valid_files = os.listdir(valid_dir)

valid_data = [] 

for filename in valid_files:
    filepath = os.path.join(valid_dir, filename)
    with open(filepath, 'rb') as f:
        rate, nsamps, seprsco = pickle.load(f)
    valid_data.append(seprsco)

# Test 
test_dir = 'nesmdb24_seprsco/test'
test_files = os.listdir(test_dir)

test_data = [] 

for filename in test_files:
    filepath = os.path.join(test_dir, filename)
    with open(filepath, 'rb') as f:
        rate, nsamps, seprsco = pickle.load(f)
    test_data.append(seprsco)


all_data = train_data + valid_data + test_data

song_lengths = []              
for song in all_data:
    song_lengths.append(len(song))
    
arr2 = np.array(song_lengths)
p30 = np.percentile(arr2, 30)
p90 = np.percentile(arr2, 90)    

all_data = [song for song in all_data if len(song) > p30 and len(song) < p90]

# Want to make sure song is long enough to be engaing 

In [5]:
size_input = 13

model = MusicLSTM(
    input_size = size_input,
    hidden_size = size_input * 4,
)  
model.load_state_dict(torch.load('best_model.pt'))
model.eval()

MusicLSTM(
  (p1_embedding): Embedding(77, 3)
  (p2_embedding): Embedding(77, 3)
  (tr_embedding): Embedding(89, 4)
  (no_embedding): Embedding(17, 3)
  (lstm1): LSTM(13, 52, batch_first=True)
  (norm1): LayerNorm((52,), eps=1e-05, elementwise_affine=True)
  (dropout1): Dropout(p=0.2, inplace=False)
  (lstm2): LSTM(52, 52, batch_first=True)
  (norm2): LayerNorm((52,), eps=1e-05, elementwise_affine=True)
  (dropout2): Dropout(p=0.2, inplace=False)
  (attention): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=52, out_features=52, bias=True)
  )
  (norm_attn): LayerNorm((52,), eps=1e-05, elementwise_affine=True)
  (out_p1): Linear(in_features=52, out_features=77, bias=True)
  (out_p2): Linear(in_features=52, out_features=77, bias=True)
  (out_tr): Linear(in_features=52, out_features=89, bias=True)
  (out_no): Linear(in_features=52, out_features=17, bias=True)
)

In [6]:
def compare_random_song():
    rate = 24
    song_id = random.randint(0, len(all_data)-1)
    song = all_data[song_id]

    
    nsamps = int(song.shape[0] / rate * 44100)
    origanal = nesmdb.convert.seprsco_to_wav((rate, nsamps, song))
    sf.write('original.wav', origanal, 44100)
    original_audio_path = 'original.wav'

    seed = song[:120]
    remapped_seed = np.column_stack([pd.Series(seed[:,0]).map(voice_data['p1']),
                                     pd.Series(seed[:,1]).map(voice_data['p2']), 
                                     pd.Series(seed[:,2]).map(voice_data['tr']), 
                                     pd.Series(seed[:,3]).map(voice_data['no'])])
    remapped_seed = remapped_seed.tolist()

    window = remapped_seed
    generated = []

    model.eval()
    with torch.no_grad():
        for step in range(len(song)-120):
            # Run the model on the current window
            predictions = model(torch.tensor(window, dtype=torch.long).unsqueeze(0))  # returns 4 outputs: p1, p2, tr, no

            # Convert each head's raw output into an actual chosen value
            next_p1 = sample_from_logits(predictions[0])
            next_p2 = sample_from_logits(predictions[1])
            next_tr = sample_from_logits(predictions[2])
            next_no = sample_from_logits(predictions[3])

            next_step = [next_p1, next_p2, next_tr, next_no]

            # Save it as part of the output
            generated.append(next_step)
    
            # Slide the window forward: drop oldest, append newest
            window = window[1:] + [next_step]
            
    full_song = remapped_seed + generated

    reverse_p1 = {v: k for k, v in voice_data['p1'].items()}
    reverse_p2 = {v: k for k, v in voice_data['p2'].items()}
    reverse_tr = {v: k for k, v in voice_data['tr'].items()}
    reverse_no = {v: k for k, v in voice_data['no'].items()}

    full_song_array = np.array(full_song)
    unmapped_song = np.column_stack([
        pd.Series(full_song_array[:,0]).map(reverse_p1),
        pd.Series(full_song_array[:,1]).map(reverse_p2),
        pd.Series(full_song_array[:,2]).map(reverse_tr),
        pd.Series(full_song_array[:,3]).map(reverse_no),
    ])

    new = nesmdb.convert.seprsco_to_wav((rate, nsamps, unmapped_song))
    sf.write('generated.wav', new, 44100)
    generated_audio_path = 'generated.wav'


    
    return original_audio_path, generated_audio_path

def cold_start_generate():
    rate = 24
    
    p1_song = []
    p2_song = []
    tr_song = []
    no_song = []
 
    for i in range(30):
        p1 = random.randint(0, len(voice_data['p1'])-1)
        p2 = random.randint(0, len(voice_data['p2'])-1)
        tr = random.randint(0, len(voice_data['tr'])-1)
        no = random.randint(0, len(voice_data['no'])-1)
        
        p1_song.append(p1)
        p2_song.append(p2)
        tr_song.append(tr)
        no_song.append(no)

    generated_song_seed = np.column_stack([p1_song,
                                           p2_song,
                                           tr_song,
                                           no_song
                                          ])


    window2 = generated_song_seed.tolist()
    generated2 = []

    model.eval()
    with torch.no_grad():
        for step in range(500):
            # Run the model on the current window
            predictions = model(torch.tensor(window2, dtype=torch.long).unsqueeze(0))  # returns 4 outputs: p1, p2, tr, no

            # Convert each head's raw output into an actual chosen value
            next_p1 = sample_from_logits(predictions[0])
            next_p2 = sample_from_logits(predictions[1])
            next_tr = sample_from_logits(predictions[2])
            next_no = sample_from_logits(predictions[3])

            next_step = [next_p1, next_p2, next_tr, next_no]

            # Save it as part of the output
            generated2.append(next_step)
    
            # Slide the window forward: drop oldest, append newest
            window2 = window2[1:] + [next_step]
            
    full_generated_song = generated2[30:]

    reverse_p1 = {v: k for k, v in voice_data['p1'].items()}
    reverse_p2 = {v: k for k, v in voice_data['p2'].items()}
    reverse_tr = {v: k for k, v in voice_data['tr'].items()}
    reverse_no = {v: k for k, v in voice_data['no'].items()}
    
    full_generated_array = np.array(full_generated_song)
    unmapped_generated_song = np.column_stack([
        pd.Series(full_generated_array[:,0]).map(reverse_p1),
        pd.Series(full_generated_array[:,1]).map(reverse_p2),
        pd.Series(full_generated_array[:,2]).map(reverse_tr),
        pd.Series(full_generated_array[:,3]).map(reverse_no),
    ])
    unmapped_generated_song[:,3] = 0 # the output was too scratchy/staticy without this set to zero, needs to be adressed in the future 

    nsamps = int(unmapped_generated_song.shape[0] / rate * 44100)
    generated_in_full = nesmdb.convert.seprsco_to_wav((rate, nsamps, unmapped_generated_song))
    sf.write('full_generated.wav', generated_in_full, 44100)
    full_generated_audio_path = 'full_generated.wav'
                                           
    return full_generated_audio_path

In [7]:
import gradio as gr

with gr.Blocks() as demo:
    btn1 = gr.Button("Compare original vs generated")
    audio_orig = gr.Audio(label="Original")
    audio_gen1 = gr.Audio(label="Generated")
    btn1.click(fn=compare_random_song, outputs=[audio_orig, audio_gen1])

    btn2 = gr.Button("Cold-start generation")
    audio_gen2 = gr.Audio(label="Generated from nothing")
    btn2.click(fn=cold_start_generate, outputs=[audio_gen2])

demo.launch()

* Running on local URL:  http://127.0.0.1:7880
* To create a public link, set `share=True` in `launch()`.
